In [ ]:
"""
Converted from IPYNB to PY
"""

**Tim Penyusun:**
- Naufal Ihsanul Islam (F1D02310084)
- Nengah Anggi Juwita Sari (F1D02310021)
- Lutfi Alfarizi (F1D02310121)

---

# S5: Perbandingan Akhir dan Sintesis Pemilihan Model
Notebook ini adalah ujung tombak dari keseluruhan eksperimen peramalan harga pangan. Di sini, kita merajut dan mengadu langsung hasil evaluasi dari keempat kandidat model (ARIMA, Random Forest Murni, PCA + RF, dan LDA + RF) yang direkam di ketiga babak pengujian. 
Hasil olahan pada halaman ini akan merumuskan kesimpulan puncak tentang model mana yang paling pantas diandalkan untuk menopang kebijakan daerah.


### 1. Persiapan Library dan Konfigurasi
Mengimpor pustaka fundamental analisis (Pandas dan Numpy), modul penggambaran visual (Matplotlib), serta perkakas penambang teks (*Regular Expression* atau `re`). Penataan gaya grafis dipoles dengan kanvas yang tajam, fon tanpa-kait (*sans-serif*) yang tegas, dan ukuran bingkai yang selaras agar diagram komparasi yang dihasilkan memancarkan aura presisi profesionalisme.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

### 2. Memuat Hasil Evaluasi (Parsing Ekstraksi Metrik)
Sel kode ini beroperasi sebagai mesin pemanen otomatis. Ia berkeliling membaca rekam jejak teks (file `.txt`) yang ditinggalkan oleh keempat model dari skrip sebelumnya, kemudian mengeruk angka-angka esensial (Jumlah Data Uji, MAE, RMSE, MAPE, $R^2$) dari kedalaman teks menggunakan pisau bedah *Regex*, lalu menyusunnya ke dalam lemari data struktural siap saji.


In [ ]:
scenarios = {
    'hasil_evaluasi_arima.txt': 'S1 - ARIMA',
    'hasil_evaluasi_rf.txt': 'S2 - RF Murni',
    'hasil_evaluasi_pca.txt': 'S3 - PCA + RF',
    'hasil_evaluasi_lda.txt': 'S4 - LDA + RF'
}

fase_data = {
    'Fase 1': [],
    'Fase 2': [],
    'Fase 3': []
}

for txt_path, model_name in scenarios.items():
    if os.path.exists(txt_path):
        with open(txt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        def extract_metrics(text_block):
            rmse_m = re.search(r'RMSE\s*:\s*Rp\s*([\d,.]+)', text_block)
            mae_m = re.search(r'MAE\s*:\s*Rp\s*([\d,.]+)', text_block)
            mape_m = re.search(r'MAPE\s*:\s*([\d.]+)%', text_block)
            r2_m = re.search(r'R2\s*:\s*([\d.]+)%', text_block)
            data_uji_m = re.search(r'Jumlah Data Uji\s*:\s*(\d+)', text_block)
            
            rmse_val = float(rmse_m.group(1).replace(',', '')) if rmse_m else 0
            mae_val = float(mae_m.group(1).replace(',', '')) if mae_m else 0
            mape_val = float(mape_m.group(1)) if mape_m else 0
            r2_val = float(r2_m.group(1)) if r2_m else 0
            data_uji = int(data_uji_m.group(1)) if data_uji_m else 0
            return data_uji, mae_val, rmse_val, mape_val, r2_val

        # Parse Fase 1
        f1_match = re.search(r'FASE 1: EVALUASI KONDISI AWAL.*?==================================================(.*?)(?:FASE 2|==================================================)', content, re.DOTALL)
        if f1_match:
            d, mae, rmse, mape, r2 = extract_metrics(f1_match.group(1))
            fase_data['Fase 1'].append({'Model': model_name, 'Data Uji': d, 'MAE (Rp)': mae, 'RMSE (Rp)': rmse, 'MAPE (%)': mape, 'R2 (%)': r2})
            
        # Parse Fase 2
        f2_match = re.search(r'FASE 2: EVALUASI.*?==================================================(.*?)(?:FASE 3|==================================================)', content, re.DOTALL)
        if f2_match:
            d, mae, rmse, mape, r2 = extract_metrics(f2_match.group(1))
            fase_data['Fase 2'].append({'Model': model_name, 'Data Uji': d, 'MAE (Rp)': mae, 'RMSE (Rp)': rmse, 'MAPE (%)': mape, 'R2 (%)': r2})
            
        # Parse Fase 3
        f3_match = re.search(r'FASE 3: EVALUASI METRIK ADIL.*?==================================================(.*?)(?:==================================================|$)', content, re.DOTALL)
        if f3_match:
            d, mae, rmse, mape, r2 = extract_metrics(f3_match.group(1))
            fase_data['Fase 3'].append({'Model': model_name, 'Data Uji': d, 'MAE (Rp)': mae, 'RMSE (Rp)': rmse, 'MAPE (%)': mape, 'R2 (%)': r2})

### 3. Arena Komparasi: Pembuktian Visual Metrik Lintas Fase

Sel eksekusi ini akan melahirkan grafik batang kembar berdampingan bersumbu ganda (*dual-axis bar chart*), yang menjajarkan persentase kelemahan tebakan (MAPE) dan jejak mutlak rupiah yang terbuang (RMSE) secara bersamaan. Di bawahnya, tersaji tabel rincian performa absolut untuk memastikan tidak ada celah interpretasi yang tertinggal.

**Cara Membaca Visualisasi Ini**: 
- **Batang Biru (Kiri/MAPE)**: Menunjuk pada persentase keliru model, semakin kerdil batangnya, semakin akurat mesinnya beradaptasi.
- **Batang Oranye (Kanan/RMSE)**: Menghitung pinalti tebakan dalam mata uang Rupiah secara langsung, kembali, semakin tenggelam batangnya, makin handal sang algoritma memeluk realita pasar.


In [ ]:
def generate_comparisons(fase_name, data_list):
    if not data_list: return
    df = pd.DataFrame(data_list)
    
    print(f"\n==========================================================================================")
    print(f"                     TABEL KOMPARASI {fase_name.upper()}")
    print(f"==========================================================================================")
    print(df.to_string(index=False))
    print(f"==========================================================================================\n")
    
    x = np.arange(len(df['Model']))
    width = 0.35
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    
    bar1 = ax1.bar(x - width/2, df['MAPE (%)'], width, label='MAPE (%)', color='#1f77b4')
    bar2 = ax2.bar(x + width/2, df['RMSE (Rp)'], width, label='RMSE (Rp)', color='#ff7f0e')
    
    ax1.set_xlabel('Model')
    ax1.set_ylabel('MAPE (%)', color='#1f77b4')
    ax2.set_ylabel('RMSE (Rp)', color='#ff7f0e')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df['Model'])
    
    for b in bar1: ax1.annotate(f'{b.get_height():.2f}%', (b.get_x() + b.get_width()/2., b.get_height()), ha='center', va='bottom', fontsize=9)
    for b in bar2: ax2.annotate(f'{b.get_height():,.0f}', (b.get_x() + b.get_width()/2., b.get_height()), ha='center', va='bottom', fontsize=9)
    
    plt.title(f'Komparasi Performa {fase_name}')
    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
    plt.tight_layout()
    plt.show()
    
    df_format = df.copy()
    df_format['MAE (Rp)'] = df_format['MAE (Rp)'].apply(lambda x: f"{x:,.0f}")
    df_format['RMSE (Rp)'] = df_format['RMSE (Rp)'].apply(lambda x: f"{x:,.0f}")
    df_format['MAPE (%)'] = df_format['MAPE (%)'].apply(lambda x: f"{x:.2f}%")
    df_format['R2 (%)'] = df_format['R2 (%)'].apply(lambda x: f"{x:.2f}%")
    
    fig, ax = plt.subplots(figsize=(12, len(df_format)*0.5 + 1.5))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=df_format.values, colLabels=df_format.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 2.5)
    
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#4CAF50')
            cell.set_text_props(weight='bold', color='white')
            
    plt.title(f'Tabel Rekapitulasi - {fase_name}', fontweight='bold', fontsize=16)
    plt.tight_layout()
    plt.show()

for fase in ['Fase 1', 'Fase 2', 'Fase 3']:
    generate_comparisons(fase, fase_data[fase])

## Analisis Komprehensif Perbandingan Akhir

Seluruh rincian grafik batang kembar dan tabel komparatif lintas fase di atas adalah saksi bisu perjalanan membongkar rapuhnya metode konvensional sekaligus pencarian kebenaran empiris dalam peramalan fluktuasi ekonomi pasaran daerah. Mari kita urai benang merah dari tebaran angka-angka hasil analitik tersebut secara mendalam.

### 1. Bedah Forensik Fase 1 (Utopia Akurasi yang Membutakan)
* **Karakteristik Visual Diagram Batang**: 
  Grafik Fase 1 menyuguhkan panorama yang luar biasa melegakan; seluruh batang biru kelam tiada yang menembus atap 10%. Semua model bertepuk dada mendeklarasikan diri piawai dengan metrik keliru di bawah radar. Mesin **Random Forest Murni (S2)** menyambar singgasana kehebatan semu di **3.23%**, membayangi model klasik **ARIMA (S1)** yang nampak gagah di **5.33%**.
* **Penjelasan Realitas Defensif**: 
  Jangan tertipu! Panorama keliru ini adalah ilusi kronis "Bocor Data" (Leakage). Pemotongan masa secara serampangan telah menghadiahkan salinan bocoran hari masa depan (2025) ke keranjang memori pelatihan mesin ML. Sang Random Forest dengan mudah merakit rumusan hapalan untuk menyalin nilai berdekatan (interpolasi), ketimbang merumuskan insting tebakan (ekstrapolasi). Sedangkan pada armada ARIMA, kemenangannya diraih dari medan uji Rata-Rata Nasional berukuran kerdil (144 set) di mana rentetan riak volatilitas gejolak provinsi sengaja dihaluskan artifisial, menjadikan ujian itu hanyalah soal kanak-kanak.

### 2. Bedah Diagnostik Fase 2 (Hantaman Keras Medan Nyata)
* **Karakteristik Visual Diagram Batang**: 
  Begitu dinding pelindung kebocoran kita tambal rapat (Ujian Kronologis Murni) dan evaluasi dipaksa menerjang medan bergejolak tingkat 38 provinsi di tahun mendatang, batang grafik biru S3 (PCA) dan S4 (LDA) melesat menembus atap seolah kiamat. Persentase ralat (*MAPE*) **PCA** hancur meroket ke **17.48%** dan **LDA** runtuh hancur-hancuran di **21.24%**! Sebaliknya, **ARIMA (S1)** bertengger kalem di **6.58%** dan **RF Murni (S2)** sukses menahan goncangan mantap di **5.73%**.
  Namun, terdapat kejanggalan matematis mencolok: Jika dilihat sekilas, tiang oranye (*RMSE*) milik ARIMA tampak menjulang menjadi yang terburuk sebesar **Rp 6.396**, mengangkangi model PCA. Mengapa error Rupiah mutlak ARIMA raksasa walau persentase keliru rata-ratanya kerdil?
* **Penjelasan Kaidah Gejolak**: 
  1. Runtuhnya LDA dan PCA adalah palu vonis bahwa memenggal ribuan serpihan identitas komoditas per provinsi demi menyederhanakan data sama fatalnya dengan membutakan mata sang peramal. Data runtut harga pasar mengandalkan kontinuitas geospasial, sebuah jiwa yang serta merta mati digilas roda peredam algoritma matematis statis ini. 
  2. "Paradoks Oranye" pada ARIMA ini adalah wujud nyata monster **Bias Skala**. Dalam kawah RMSE agregat global, meleset 5% saat menebak rentang Daging Sapi seharga ratusan ribu menyumbang beban penalti ribuan rupiah berlipat yang mencemari skor mutlak model, sungguh tidak mencerminkan daya tahannya yang amat presisi di atas lapangan pangan pokok.

### 3. Bedah Klinis Fase 3 (Puncak Keadilan Metrik)
* **Karakteristik Visual Diagram Batang**: 
  Kita lalu mencuci bersih seluruh model dari hegemoni Bias Skala. Melalui pemisahan perhitungan evaluasi tersendiri dari lorong kasta tiap komoditas, kita mencabut dominasi harga selangit. Saksikan efek magisnya pada tiang RMSE (Oranye) yang kini kempis rata. Pinalti **ARIMA** menciut masif 45% menjadi tinggal **Rp 3.514**. Angka RMSE para kandidat ini akhirnya merepresentasikan level ralat rupah sejati tanpa polusi inflasi semu, memapankan kedudukan **RF Murni (S2)** dengan RMSE istimewa **Rp 2.548**.

### 4. Analisis Jatuhnya Klaim Nilai Korelasi ($R^2$)
Kehadiran Fase 3 tak luput meruntuhkan kebohongan absolut persentase kapabilitas model ($R^2$). 
* **ARIMA**: $R^2$ berinflasi **95.48%** (Fase 2) menapaki tanah kenyataan menjadi **58.86%** (Fase 3)
* **RF Murni**: $R^2$ angkuh **97.62%** (Fase 2) mendarat meyakinkan di pijakan **79.28%** (Fase 3)
* **PCA + RF**: $R^2$ menjebak **95.78%** (Fase 2) tersungkur habis di angka tragis **35.46%** (Fase 3)
* **LDA + RF**: $R^2$ raksasa **95.44%** (Fase 2) terkapar di dasar lembah kehinaan **14.70%** (Fase 3)

Nilai raksasa ~95% pada fase terdahulu ditimbulkan akibat varians target dihitung global. Pergerakan variasi harga super murah (Garam) ke harga elitis (Sapi) menjadi pembagi denominator raksasa yang mengakali rumus. Seiring perumusan adil murni per sekat komoditas, kebesaran skor RF di 79.28% menobatkan secara sahih bahwa algoritma rimbun acak sanggup membaca empat perlima dari keseluruhan rumitnya keragaman gejolak perilaku perputaran pasar pangan tingkat provinsi.

---

## Konklusi Rekomendasi Eksekutif Pemilihan Senjata Model

1. **Model Takhta Presisi**: Predikat pemuncak kecerdasan prediksi sah digenggam oleh **S2 - Random Forest Murni** (MAPE cemerlang **5.79%** dengan kemujaraban interpretasi relasi $R^2$ di atas awan, **79.28%**). Konstruksi cabang hutannya mampu mencium aroma gelagat non-linear pergerakan tersembunyi komoditas-komoditas rumit secara brilian. Syarat pengaplikasian model ini mutlak: Model takhta ini tak boleh dikebiri fiturnya oleh operasi matematis reduksi apapun.

2. **Model Takhta Stabilitas Klasik**: Posisi model paling gigih direngkuh **S1 - ARIMA**. Walau merupakan mesin tua beralur silinder tunggal (univariat), model ini sanggup membelah waktu tanpa kepanikan (MAPE amat santun **6.44%**, $R^2$ terukur kokoh **58.86%**). ARIMA membuktikan superioritas arsitektur hematnya; sanggup hidup, bertahan dan menaklukan badai spasial keragaman lintas wilayah tanpa menuntut pakan beban dimensi fitur melimpah. Model ini jawaranya instansi di kawasan dengan infrastruktur sumber daya analitik pas-pasan namun menuntut hasil konstan konsisten terpercaya.

3. **Deklarasi Larangan Reduksi**: Hasil visum menelanjangi ketidakcocokan kronis penggunaan pilar PCA (S3) dan pemenggal kelompok kategori LDA (S4) pada arena prediksi ekonomi waktu harga komoditas pangan ini. Reduksi dimensi terbukti sepenuhnya cacat saat disuruh memelihara benang kontinuitas harga yang bergulir temporal; menggunakan operasi pemenggalan variabel ini murni sebuah resep jitu menjamin gagalnya rumusan kebijakan masa depan.
